First, we read the csv file.

In [ ]:
import pandas as pd

yellow_df = pd.read_csv("yellow_tripdata_2025-08.csv")

print(yellow_df.shape)      
print(yellow_df.columns)    
print(yellow_df.head())     


/var/folders/rq/d1f380z1435dgk98n9b9hwz40000gn/T/ipykernel_97432/1971641118.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  yellow_df = pd.read_csv("yellow_tripdata_2025-08.csv")


(3574091, 20)
Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2025-08-01 00:52:23   2025-08-01 01:12:20              1.0   
1         2  2025-08-01 00:03:01   2025-08-01 00:15:33              2.0   
2         7  2025-08-01 00:24:38   2025-08-01 00:24:38              2.0   
3         7  2025-08-01 00:48:19   2025-08-01 00:48:19              1.0   
4         2  2025-08-01 00:25:34   2025-08-01 00:33:18              1.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           8.44         1.0               

In [ ]:
yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-08-01 00:52:23,2025-08-01 01:12:20,1.0,8.44,1.0,N,138,141,1,33.8,6.0,0.5,5.00,6.94,1.0,57.49,2.5,1.75,0.00
1,2,2025-08-01 00:03:01,2025-08-01 00:15:33,2.0,4.98,1.0,N,138,193,1,21.2,6.0,0.5,0.00,0.00,1.0,30.45,0.0,1.75,0.00
2,7,2025-08-01 00:24:38,2025-08-01 00:24:38,2.0,1.89,1.0,N,249,45,1,14.2,0.0,0.5,3.99,0.00,1.0,23.94,2.5,0.00,0.75
3,7,2025-08-01 00:48:19,2025-08-01 00:48:19,1.0,2.35,1.0,N,79,229,1,11.4,0.0,0.5,3.43,0.00,1.0,20.58,2.5,0.00,0.75
4,2,2025-08-01 00:25:34,2025-08-01 00:33:18,1.0,2.14,1.0,N,43,48,1,11.4,1.0,0.5,2.57,0.00,1.0,19.72,2.5,0.00,0.75


Here, we are projecting the clolumns that we need for this project, which are "tpep_pickup_datetime","tpep_dropoff_datetime","PULocationID","DOLocationID","total_amount"

In [ ]:
yellow_df = yellow_df[["tpep_pickup_datetime","tpep_dropoff_datetime","PULocationID","DOLocationID","total_amount"]]

In [ ]:
yellow_df

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,total_amount
0,2025-08-01 00:52:23,2025-08-01 01:12:20,138,141,57.49
1,2025-08-01 00:03:01,2025-08-01 00:15:33,138,193,30.45
2,2025-08-01 00:24:38,2025-08-01 00:24:38,249,45,23.94
3,2025-08-01 00:48:19,2025-08-01 00:48:19,79,229,20.58
4,2025-08-01 00:25:34,2025-08-01 00:33:18,43,48,19.72
...,...,...,...,...,...
3574086,2025-08-31 23:40:30,2025-08-31 23:58:48,65,228,23.03
3574087,2025-08-31 23:10:26,2025-08-31 23:29:01,236,148,26.53
3574088,2025-08-31 23:04:21,2025-08-31 23:25:21,148,48,5.83
3574089,2025-08-31 23:44:26,2025-08-31 23:44:42,107,107,17.42


Here we want to double check if all the missing value data have been cleaned.

In [ ]:
# check missing values
yellow_df.isna().sum()

# check if we have NaN
yellow_df.isna().any()

tpep_pickup_datetime     False
tpep_dropoff_datetime    False
PULocationID             False
DOLocationID             False
total_amount             False
dtype: bool

Here we convert all the datetime column data into correct format of time and check the NaT values for the datetime.

In [ ]:
# convert to datetime 
yellow_df["tpep_pickup_datetime"] = pd.to_datetime(yellow_df["tpep_pickup_datetime"], errors="coerce")
yellow_df["tpep_dropoff_datetime"] = pd.to_datetime(yellow_df["tpep_dropoff_datetime"], errors="coerce")

# check how many NaT do we have
yellow_df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]].isna().sum()


tpep_pickup_datetime     0
tpep_dropoff_datetime    0
dtype: int64

We are also dropping the rows that are not correct logically, which are the rows that have dropoff earlier than pickup.

In [ ]:
# dropoff should be later than pickup 
invalid_time = yellow_df[yellow_df["tpep_dropoff_datetime"] <= yellow_df["tpep_pickup_datetime"]]
len(invalid_time)  # rows abnormal?

invalid_time

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,total_amount
2,2025-08-01 00:24:38,2025-08-01 00:24:38,249,45,23.94
3,2025-08-01 00:48:19,2025-08-01 00:48:19,79,229,20.58
48,2025-08-01 00:56:52,2025-08-01 00:56:52,144,264,11.55
139,2025-08-01 00:18:29,2025-08-01 00:18:29,246,90,12.95
140,2025-08-01 00:36:13,2025-08-01 00:36:13,50,50,14.10
...,...,...,...,...,...
3311884,2025-08-22 20:48:00,2025-08-22 20:48:00,48,48,9.71
3348076,2025-08-23 20:19:00,2025-08-23 20:19:00,186,186,4.28
3382239,2025-08-24 18:26:00,2025-08-24 18:26:00,17,17,21.81
3454906,2025-08-28 02:48:00,2025-08-28 02:48:00,112,112,7.06


In [ ]:
yellow_df = yellow_df[yellow_df["tpep_dropoff_datetime"] > yellow_df["tpep_pickup_datetime"]]

Here, we need to drop the rows that have a invalid trip time, which are the trips that are less than 5 minutes and more than 180 minutes.

In [ ]:
# trip time
yellow_df["trip_minutes"] = (
    (yellow_df["tpep_dropoff_datetime"] - yellow_df["tpep_pickup_datetime"]).dt.total_seconds() / 60
)

yellow_df["trip_minutes"].describe()

# extreme values of trip time
yellow_df[(yellow_df["trip_minutes"] < 5) | (yellow_df["trip_minutes"] > 180)]


/var/folders/rq/d1f380z1435dgk98n9b9hwz40000gn/T/ipykernel_97432/703640050.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yellow_df["trip_minutes"] = (


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,total_amount,trip_minutes
7,2025-08-01 00:12:10,2025-08-01 00:15:58,163,236,16.38,3.800000
8,2025-08-01 00:05:30,2025-08-01 00:09:19,263,262,10.80,3.816667
17,2025-08-01 00:21:50,2025-08-01 00:24:20,100,170,12.18,2.500000
19,2025-08-01 00:51:39,2025-08-01 00:55:32,263,236,12.96,3.883333
22,2025-08-01 00:37:02,2025-08-01 00:41:22,114,211,14.70,4.333333
...,...,...,...,...,...,...
3574021,2025-08-31 23:16:07,2025-08-31 23:20:44,37,37,12.23,4.616667
3574022,2025-08-31 23:19:37,2025-08-31 23:24:16,216,10,10.87,4.650000
3574053,2025-08-31 23:58:50,2025-08-31 23:59:02,246,246,13.48,0.200000
3574060,2025-08-31 23:17:43,2025-08-31 23:20:44,148,114,6.51,3.016667


In [ ]:
yellow_df = yellow_df[(yellow_df["trip_minutes"] > 5) & (yellow_df["trip_minutes"] < 180)]

In [ ]:
yellow_df

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,total_amount,trip_minutes
0,2025-08-01 00:52:23,2025-08-01 01:12:20,138,141,57.49,19.950000
1,2025-08-01 00:03:01,2025-08-01 00:15:33,138,193,30.45,12.533333
4,2025-08-01 00:25:34,2025-08-01 00:33:18,43,48,19.72,7.733333
5,2025-08-01 00:16:36,2025-08-01 00:33:41,114,230,24.15,17.083333
6,2025-08-01 00:56:02,2025-08-01 01:15:37,163,13,31.45,19.583333
...,...,...,...,...,...,...
3574085,2025-08-31 23:57:16,2025-09-01 00:04:19,90,230,17.00,7.050000
3574086,2025-08-31 23:40:30,2025-08-31 23:58:48,65,228,23.03,18.300000
3574087,2025-08-31 23:10:26,2025-08-31 23:29:01,236,148,26.53,18.583333
3574088,2025-08-31 23:04:21,2025-08-31 23:25:21,148,48,5.83,21.000000


Here, we also need to filter the data based on the total amount of fare, which can not be a negative value.

In [ ]:
# .describe
yellow_df["total_amount"].describe()

# negative value?
# yellow_df[(yellow_df["total_amount"] < 0) | (yellow_df["total_amount"] > 500)]
yellow_df = yellow_df[yellow_df["total_amount"] > 0]


In [ ]:
yellow_df

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,total_amount,trip_minutes
0,2025-08-01 00:52:23,2025-08-01 01:12:20,138,141,57.49,19.950000
1,2025-08-01 00:03:01,2025-08-01 00:15:33,138,193,30.45,12.533333
4,2025-08-01 00:25:34,2025-08-01 00:33:18,43,48,19.72,7.733333
5,2025-08-01 00:16:36,2025-08-01 00:33:41,114,230,24.15,17.083333
6,2025-08-01 00:56:02,2025-08-01 01:15:37,163,13,31.45,19.583333
...,...,...,...,...,...,...
3574085,2025-08-31 23:57:16,2025-09-01 00:04:19,90,230,17.00,7.050000
3574086,2025-08-31 23:40:30,2025-08-31 23:58:48,65,228,23.03,18.300000
3574087,2025-08-31 23:10:26,2025-08-31 23:29:01,236,148,26.53,18.583333
3574088,2025-08-31 23:04:21,2025-08-31 23:25:21,148,48,5.83,21.000000


And for the LocationID, we only want the ID to be from 0 to 263 since the rest are all Unknown, so we need to drop these invalid values.

In [ ]:
# ID should be integer&non-negative, also we have only 265 locations
yellow_df[["PULocationID", "DOLocationID"]].describe()

invalid_loc = yellow_df[
    (yellow_df["PULocationID"] <= 0) |
    (yellow_df["DOLocationID"] <= 0) |
    (yellow_df["PULocationID"] > 263) |
    (yellow_df["DOLocationID"] > 263)
]
len(invalid_loc), invalid_loc.head()

(0,
 Empty DataFrame
 Columns: [tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID, total_amount, trip_minutes]
 Index: [])

In [ ]:
yellow_df = yellow_df[
    (yellow_df["PULocationID"] > 0) &
    (yellow_df["DOLocationID"] > 0) &
    (yellow_df["PULocationID"] <= 263) &
    (yellow_df["DOLocationID"] <= 263)
]

In [ ]:
yellow_df.drop(columns=["trip_minutes"])

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,total_amount
0,2025-08-01 00:52:23,2025-08-01 01:12:20,138,141,57.49
1,2025-08-01 00:03:01,2025-08-01 00:15:33,138,193,30.45
4,2025-08-01 00:25:34,2025-08-01 00:33:18,43,48,19.72
5,2025-08-01 00:16:36,2025-08-01 00:33:41,114,230,24.15
6,2025-08-01 00:56:02,2025-08-01 01:15:37,163,13,31.45
...,...,...,...,...,...
3574085,2025-08-31 23:57:16,2025-09-01 00:04:19,90,230,17.00
3574086,2025-08-31 23:40:30,2025-08-31 23:58:48,65,228,23.03
3574087,2025-08-31 23:10:26,2025-08-31 23:29:01,236,148,26.53
3574088,2025-08-31 23:04:21,2025-08-31 23:25:21,148,48,5.83


In [ ]:
yellow_df = yellow_df.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "PULocationID": "pickup_location",
    "DOLocationID": "dropoff_location",
    "total_amount": "total_amount"
})
print('ok')

ok


In [ ]:
yellow_df.to_csv('yellow_cleaned_df_1114.csv')